In [1]:
import pandas as pd

income_stmt = pd.read_csv("Data/income_statement.csv", index_col=0)
balance_sheet = pd.read_csv("data/balance_sheet.csv", index_col=0)
cash_flow = pd.read_csv("data/cash_flow.csv", index_col=0)

In [2]:
print(income_stmt.columns.tolist())
print(balance_sheet.columns.tolist())
print(cash_flow.columns.tolist())

['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Net Non Operating Interest Income Expense', 'Interest Expense Non Operating', 'Interest Income Non Operating', 'Operating Income', 'Operating Expense', 'Research And Development', 'Selling General And Administration', 'Gross Profit', 'Cost Of R

In [3]:
# convert all values to numeric, coerce errors to NaN
income_stmt = income_stmt.apply(pd.to_numeric, errors='coerce')
balance_sheet = balance_sheet.apply(pd.to_numeric, errors='coerce')
cash_flow = cash_flow.apply(pd.to_numeric, errors='coerce')

In [4]:
# income statement metrics
def calc_income_metrics(df):
    metrics = pd.DataFrame(index=df.index)
    metrics['Revenue'] = df['Total Revenue']
    metrics['Gross Profit'] = df['Gross Profit']
    metrics['EBIT'] = df['EBIT']
    metrics['EBITDA'] = df['EBITDA']
    metrics['Net Income'] = df['Net Income']
    
    metrics['Gross Margin %'] = df['Gross Profit'] / df['Total Revenue'] * 100
    metrics['EBIT Margin %'] = df['EBIT'] / df['Total Revenue'] * 100
    metrics['Net Margin %'] = df['Net Income'] / df['Total Revenue'] * 100

    metrics = metrics.dropna(how='all')
    return metrics.round(2)

income_metrics = calc_income_metrics(income_stmt)
print(income_metrics)

                 Revenue  Gross Profit          EBIT        EBITDA  \
2025-09-30  4.161610e+11  1.952010e+11  1.330500e+11  1.447480e+11   
2024-09-30  3.910350e+11  1.806830e+11  1.232160e+11  1.346610e+11   
2023-09-30  3.832850e+11  1.691480e+11  1.143010e+11  1.258200e+11   
2022-09-30  3.943280e+11  1.707820e+11  1.194370e+11  1.305410e+11   

              Net Income  Gross Margin %  EBIT Margin %  Net Margin %  
2025-09-30  1.120100e+11           46.91          31.97         26.92  
2024-09-30  9.373600e+10           46.21          31.51         23.97  
2023-09-30  9.699500e+10           44.13          29.82         25.31  
2022-09-30  9.980300e+10           43.31          30.29         25.31  


In [5]:
# balance sheet metrics
def calc_balance_metrics(df):
    metrics = pd.DataFrame(index=df.index)
    
    metrics['Total Assets'] = df['Total Assets']
    metrics['Total Liabilities'] = df['Total Liabilities Net Minority Interest']
    metrics['Total Equity'] = df['Stockholders Equity']
    metrics['Total Debt'] = df['Total Debt']
    metrics['Cash'] = df['Cash And Cash Equivalents']
    metrics['Net Debt'] = df['Net Debt']

    # check: shd be close to 0 if balance sheet balance_sheet
    metrics['Check(A-L-E)'] = (df['Total Assets'] 
                               - df['Total Liabilities Net Minority Interest'] 
                               - df['Stockholders Equity'])
    
    metrics = metrics.dropna(how='all')
    return metrics.round(2)

balance_metrics = calc_balance_metrics(balance_sheet)
print(balance_metrics)

            Total Assets  Total Liabilities  Total Equity    Total Debt  \
2025-09-30  3.592410e+11       2.855080e+11  7.373300e+10  9.865700e+10   
2024-09-30  3.649800e+11       3.080300e+11  5.695000e+10  1.066290e+11   
2023-09-30  3.525830e+11       2.904370e+11  6.214600e+10  1.110880e+11   
2022-09-30  3.527550e+11       3.020830e+11  5.067200e+10  1.324800e+11   

                    Cash      Net Debt  Check(A-L-E)  
2025-09-30  3.593400e+10  6.272300e+10           0.0  
2024-09-30  2.994300e+10  7.668600e+10           0.0  
2023-09-30  2.996500e+10  8.112300e+10           0.0  
2022-09-30  2.364600e+10  9.642300e+10           0.0  


In [6]:
# cash flow metrics
def calc_cashflow_metrics(df):
    metrics = pd.DataFrame(index=df.index)

    metrics['Operating CF'] = df['Operating Cash Flow']
    metrics['Capex'] = df['Capital Expenditure']
    metrics['Free Cash Flow'] = df['Free Cash Flow']
    metrics['D&A'] = df['Depreciation And Amortization']

    metrics = metrics.dropna(how='all')
    return metrics.round(2)

cf_metrics = calc_cashflow_metrics(cash_flow)
print(cf_metrics)

            Operating CF         Capex  Free Cash Flow           D&A
2025-09-30  1.114820e+11 -1.271500e+10    9.876700e+10  1.169800e+10
2024-09-30  1.182540e+11 -9.447000e+09    1.088070e+11  1.144500e+10
2023-09-30  1.105430e+11 -1.095900e+10    9.958400e+10  1.151900e+10
2022-09-30  1.221510e+11 -1.070800e+10    1.114430e+11  1.110400e+10


In [7]:
# discounted cash flow (dcf) valuation
def calc_dcf(cf_df, wacc=0.09, terminal_growth=0.03, forecast_years=5):
    """
    wacc = discount rate (9% is a reasonable starting point for Apple)
    terminal_growth = long_term FCF growth rate (3% ≈ GDP growth, conservative)
    forecast_years = how many years to project forward (in this case, projecting 5 years into future)
    """
    """
    core intuition: "if i could see all the cash Apple will ever generate, and i discount to today's $,
    what is the company worth right now?"
    """
    # use most recent year's free cash flow (FCF) as base
    fcf_base = cf_df['Free Cash Flow'].iloc[0]
    
    # estimate FCF growth rate from historical data (average YOY growth)
    fcf_series = cf_df['Free Cash Flow'].dropna()
    fcf_growth = fcf_series.pct_change(-1).mean()    # avg historical growth
    fcf_growth = max(fcf_growth, 0.05)    # floor at 5% minimum growth 

    print(f"Base FCF: ${fcf_base/1e9:.2f}B")
    print(f"FCF Growth: {fcf_growth*100:.1f}%")
    print(f"WACC: {wacc*100:.1f}%")
    print(f"Terminal Growth: {terminal_growth*100:.1f}%")

    # project FCFs forward
    projected_fcfs = []
    for year in range(1, forecast_years+1):
        projected_fcf = fcf_base * (1 + fcf_growth) ** year
        discounted_fcf = projected_fcf / (1 + wacc) ** year  # converts to today's value
        projected_fcfs.append({
            'Year': f'Year {year}',
            'Projected FCF': round(projected_fcf/1e9, 2),
            'Discounted FCF': round(discounted_fcf/1e9, 2)
        })
    
    dcf_df = pd.DataFrame(projected_fcfs)

    # terminal value: captures all cash flows beyond forecast period 
    fcf_final = fcf_base * (1 + fcf_growth) ** forecast_years    # the year 5 projected FCF
    terminal_value = fcf_final * (1 + terminal_growth) / (wacc - terminal_growth)   # Gordon Growth Model
    pv_terminal = terminal_value / (1 + wacc) ** forecast_years   # discounts terminal value back to today

    # intrinsic value = sum of discounted FCFs + PV of terminal value
    pv_fcfs = sum([row['Discounted FCF'] for row in projected_fcfs])
    intrinsic_value = pv_fcfs + (pv_terminal / 1e9)

    print(f"\nProjected FCFs:")
    print(dcf_df.to_string(index=False))
    print(f"\nTerminal Value (PV): ${pv_terminal/1e9:.2f}B")
    print(f"PV of FCFs: ${pv_fcfs:.2f}B")
    print(f"DCF Intrinsic Value: ${intrinsic_value:.2f}B")

    return dcf_df, intrinsic_value

dcf_table, intrinsic_value = calc_dcf(cf_metrics)

Base FCF: $98.77B
FCF Growth: 5.0%
WACC: 9.0%
Terminal Growth: 3.0%

Projected FCFs:
  Year  Projected FCF  Discounted FCF
Year 1         103.71           95.14
Year 2         108.89           91.65
Year 3         114.34           88.29
Year 4         120.05           85.05
Year 5         126.05           81.93

Terminal Value (PV): $1406.41B
PV of FCFs: $442.06B
DCF Intrinsic Value: $1848.47B


In [8]:
# openpyxl output

# install & import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, numbers
from openpyxl.styles.numbers import FORMAT_NUMBER_COMMA_SEPARATED1
from openpyxl.utils.dataframe import dataframe_to_rows

In [9]:
# colours
HEADER_FILL = PatternFill("solid", fgColor="1F4E79")  # dark blue
SUBHEAD_FILL = PatternFill("solid", fgColor="2E75B6")  # medium blue
ALT_ROW_FILL = PatternFill("solid", fgColor="EBF3FB")  # light blue

# fonts
HEADER_FONT = Font(bold=True, color="FFFFFF", size=11)
SUBHEAD_FONT = Font(bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(size=10)

# alignment
CENTER = Alignment(horizontal='center')
RIGHT = Alignment(horizontal="right")

In [10]:
# formatting function 
def format_sheet(ws, df, currency_cols=None, pct_cols=None):
    """
    ws = the worksheet object
    currency_cols = the list of column names to format as $#,##0
    pct_cols = list of column names to format as 0.00%
    """

    # columns widths
    ws.column_dimensions['A'].width = 22  # row labels
    for col in ws.iter_cols(min_col=2, max_col=ws.max_column):
        ws.column_dimensions[col[0].column_letter].width = 18
    
    for i, row in enumerate(ws.iter_rows()):
        for j, cell in enumerate(row):
            # header row (row 1)
            if i == 0:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
                cell.alignment = CENTER

            # index column (column A, skip header)
            elif j == 0:
                cell.fill = SUBHEAD_FILL
                cell.font = SUBHEAD_FONT
                cell.alignment = CENTER
            
            # alternating row fills
            else:
                cell.font = BODY_FONT
                cell.alignment = RIGHT
                if i % 2:
                    cell.fill = ALT_ROW_FILL
    
    # apply number formats by column name :)
    for col_idx, col_name in enumerate(df.columns, start=2):
        col_letter = ws.cell(row=1, column=col_idx).column_letter
        for row_idx in range(2, ws.max_row + 1):
            cell = ws[f"{col_letter}{row_idx}"]
            if currency_cols and col_name in currency_cols:
                cell.number_format = '$#,##0.0,, "B"'   # shows billions
            elif pct_cols and col_name in pct_cols:
                cell.number_format = '0.00"%"'

In [13]:
wb = Workbook()

wb.remove(wb.active)  # removing default empty sheet


# income statement
ws_is = wb.create_sheet("Income Statement")
for row in dataframe_to_rows(income_metrics, index=True, header=True):
    ws_is.append(row)
ws_is.delete_rows(2)
format_sheet(
    ws_is, income_metrics,
    currency_cols=['Revenue', 'Gross Profit', 'EBIT', 'EBITDA', 'Net Income'],
    pct_cols=['Gross Margin %', 'EBIT Margin %', 'Net Margin %']
)

# balance sheet
ws_bs = wb.create_sheet("Balance Sheet")
for row in dataframe_to_rows(balance_metrics, index=True, header=True):
    ws_bs.append(row)
ws_bs.delete_rows(2)
format_sheet(
    ws_bs, balance_metrics,
    currency_cols=['Total Assets', 'Total Liabilities', 'Total Equity',
                   'Total Debt', 'Cash', 'Net Debt']
)

# cash flow sheet 
ws_cf = wb.create_sheet("Cash Flow")
for row in dataframe_to_rows(cf_metrics, index=True, header=True):
    ws_cf.append(row)
ws_cf.delete_rows(2)
format_sheet(
    ws_cf, cf_metrics,
    currency_cols=['Operating CF', 'Capex', 'Free Cash Flow', 'D&A']
)

# DCF valuation sheet
ws_dcf = wb.create_sheet("DCF Valuation")
for row in dataframe_to_rows(dcf_table, index=True, header=True):
    ws_dcf.append(row)
ws_dcf.delete_rows(2)    
format_sheet(
    ws_dcf, dcf_table,
    currency_cols=['Projected FCF', 'Discounted FCF']
)
for row in ws_dcf.iter_rows(min_row=2, min_col=2, max_col=ws_dcf.max_column):
    for cell in row:
        if isinstance(cell.value, (int, float)):
            cell.number_format = '"$"#,##0.00"B"'

# summary sheet
ws_sum = wb.create_sheet("Summary", 0)    # the 0 puts it first
ws_sum['B2'] = 'Apple Inc. - Financial Model Summary'
ws_sum['B2'].font = Font(bold=True, size=14, color="1F4E79")

ws_sum['B4'] = "Metric"
ws_sum['C4'] = "FY2025"
ws_sum['D4'] = "FY2024"

for cell in [ws_sum['B4'], ws_sum['C4'], ws_sum['D4']]:
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = CENTER

summary_rows = {
    ("Revenue", income_metrics.loc['2025-09-30', 'Revenue'], income_metrics.loc['2024-09-30', 'Revenue']),
    ("Gross Margin %", income_metrics.loc['2025-09-30', 'Gross Margin %'], income_metrics.loc['2024-09-30', 'Gross Margin %']),
    ("EBIT Margin %", income_metrics.loc['2025-09-30', 'EBIT Margin %'], income_metrics.loc['2024-09-30', 'EBIT Margin %']),
    ("Net Income", income_metrics.loc['2025-09-30', 'Net Income'], income_metrics.loc['2024-09-30', 'Net Income']),
    ("Free Cash Flow", cf_metrics.loc['2025-09-30', 'Free Cash Flow'], cf_metrics.loc['2024-09-30', 'Free Cash Flow']),
    ("Total Debt", balance_metrics.loc['2025-09-30', 'Total Debt'], balance_metrics.loc['2024-09-30', 'Total Debt']),
    ("Net Debt", balance_metrics.loc['2025-09-30', 'Net Debt'], balance_metrics.loc['2024-09-30', 'Net Debt']),
    ("DCF Intrinsic Value ($B)", intrinsic_value, None)
}

pct_rows =  {"Gross Margin %", "EBIT Margin %"}

for i, (label, val1, val2) in enumerate(summary_rows, start=5):
    ws_sum[f'B{i}'] = label
    ws_sum[f'B{i}'].font = Font(bold=True, size=10)
    ws_sum[f'C{i}'] = val1
    ws_sum[f'D{i}'] = val2 if val2 is not None else "—"

    if label == "DCF Intrinsic Value ($B)":
        fmt = '"$"#,##0.00"B"'
    elif label in pct_rows:
        fmt = '0.00"%"'
    else:
        fmt = '$#,##0.0,,"B"'
    ws_sum[f'C{i}'].number_format = fmt

    if val2:
        ws_sum[f'D{i}'].number_format = fmt

    if i % 2 == 0:
        for col in ['B', 'C', 'D']:
            ws_sum[f'{col}{i}'].fill = ALT_ROW_FILL

ws_sum.column_dimensions['B'].width = 28
ws_sum.column_dimensions['C'].width = 26
ws_sum.column_dimensions['D'].width = 16

# revenue & FCF chart
from openpyxl.chart import BarChart, Reference, Series
ws_chart = wb.create_sheet("Chart")

years = list(income_metrics.index)
revenues = list(income_metrics['Revenue'])
fcfs = list (cf_metrics['Free Cash Flow'])

ws_chart['A1'] = "Year"
ws_chart['B1'] = "Revenue ($B)"
ws_chart['C1'] = "Free Cash Flow ($B)"

for i, (yr, rev, fcf) in enumerate(zip(years, revenues, fcfs), start=2):
    ws_chart[f'A{i}'] = str(yr[:4])
    ws_chart[f'B{i}'] = round(float(rev) / 1e9, 1)
    ws_chart[f'C{i}'] = round(float(fcf) / 1e9, 1)
    
chart = BarChart()
chart.type = "col"
chart.title = "Apple - Revenue vs Free Cash Flow"
chart.y_axis.title = "USD Billions"
chart.x_axis.title = "Fiscal Year"
chart.style = 10
chart.width = 20
chart.height = 12
chart.grouping = 'clustered'

data = Reference(ws_chart, min_col=2, max_col=3, min_row=1, max_row=len(years)+1)
cats = Reference(ws_chart, min_col=1, min_row=2, max_row=len(years)+1)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)

ws_chart.add_chart(chart, "E2")

wb.save("financial_model.xlsx")
print("financial_model.xlsx saved!")

financial_model.xlsx saved!
